In [1]:
from importlib import reload

from collections import Counter

import math
import sage.combinat.partitions as partitions
import sage.libs.lrcalc.lrcalc as lrcalc
from sage.all import *

from sage.misc.latex import LatexExpr
from sage.combinat.composition import Compositions

all_partitions = partitions.ZS1_iterator
lrcoeff = lrcalc.lrcoef_unsafe



In [2]:
def degree(partition):
    return sum(partition)

def compare_by_degree(p1, p2):
    d1, d2 = degree(p1), degree(p2)
    if d1 < d2:
        return -1
    elif d1 > d2:
        return 1
    return 0


In [3]:
degree((3,))

3

In [4]:

def generate_partitions(n, max_part=None):
    """Generate all partitions of n."""
    if n == 0:
        yield ()
    else:
        if max_part is None or max_part > n:
            max_part = n
        for first in range(max_part, 0, -1):
            for rest in generate_partitions(n - first, first):
                yield (first,) + rest

p = (3, 1) 
n=5
parts = list(generate_partitions(5))
print(parts)




[(5,), (4, 1), (3, 2), (3, 1, 1), (2, 2, 1), (2, 1, 1, 1), (1, 1, 1, 1, 1)]


In [5]:
def subset_partitions(partition):
    """Return all partitions with degree <= degree(partition)."""
    d = degree(partition)
    result = []
    for n in range(d + 1):
        result.extend(generate_partitions(n))
    return result


subset_partitions((3,1))

[(),
 (1,),
 (2,),
 (1, 1),
 (3,),
 (2, 1),
 (1, 1, 1),
 (4,),
 (3, 1),
 (2, 2),
 (2, 1, 1),
 (1, 1, 1, 1)]

In [6]:

print("Degree of p:", degree(p))

q = (1, 1)
print("Compare p and q:", compare_by_degree(p, q))  # 0 means equal

subset = subset_partitions(p)
print(f"All partitions with degree ≤ {degree(p)}:")
print(subset)


Degree of p: 4
Compare p and q: 1
All partitions with degree ≤ 4:
[(), (1,), (2,), (1, 1), (3,), (2, 1), (1, 1, 1), (4,), (3, 1), (2, 2), (2, 1, 1), (1, 1, 1, 1)]


# Calculating 2 LW Coefficient Manually Using COmbinatorics

In [7]:

from itertools import permutations

def is_partition(p):
    return all(p[i] >= p[i+1] for i in range(len(p)-1))

def subtract_partitions(lam, mu):
    """Return skew shape lam/mu as list of row lengths."""
    if len(mu) > len(lam) or any(mu[i] > lam[i] for i in range(len(mu))):
        return None
    skew = [lam[i] - (mu[i] if i < len(mu) else 0) for i in range(len(lam))]
    return skew

def weight_to_list(weight):
    """Expand weight partition to list, e.g. (2,1) -> [1,1,2]."""
    res = []
    for i, m in enumerate(weight):
        res += [i+1]*m
    return res

def is_yamanouchi(word):
    """Check lattice word (Yamanouchi) condition."""
    counts = {}
    for x in word:
        counts[x] = counts.get(x, 0) + 1
        for y in range(1, x):
            if counts.get(y, 0) < counts[x]:
                return False
    return True

def lrcoefP(mu, nu, lam):
    """Compute N^{lam}_{mu,nu} using Yamanouchi condition."""
    skew = subtract_partitions(lam, mu)
    if skew is None:
        return 0
    num_boxes = sum(skew)
    if num_boxes != sum(nu):
        return 0
    if mu == () and degree(nu) == degree(lam) and nu != lam:
        return 0
    if nu == () and degree(mu) == degree(lam) and mu != lam:
        return 0
    # naive enumeration for small cases
    entries = weight_to_list(nu)
    c = 0
    for perm in set(permutations(entries)):
        if is_yamanouchi(perm):
            c += 1
    return c

In [8]:
# Tests LW 2
print(lrcoefP((2,), (1,1), (4,)))    
print(lrcoeff((4,), (2,), (1,1)))
print(lrcalc.lrcoef((1,), (1,), (2,)))

# tests 2




1
0
0


In [9]:
def solveOne(k, k1, k2, k3, k4):

    if k2 > k1 or k4 > k3:
        return []

    solutions = []


    print(f'Start of iteration for Delta (from 0 to k):\n')
    for x1 in range(k + 1):
        print(f'======= The {x1}-th ITERATION FOR DELTA: ========\n')
        print(f'    Value of delta = {x1}')

        max_x2 = math.floor((k - x1))
        range_Max = math.ceil((max_x2)/2 + 1)
        
        print(f'    Maximum Gamma = {max_x2}\n')

        print(f'    Start of iteration for Gamma (from 0 to ceil(floor(k - delta)/2) + 1): \n' )
        
        for x2 in range(range_Max):
            print(f'------- {x2}-th iteration of Gamma: --------\n')
            print(f'        Value of Gamma = {x2}')



            if range_Max == 0:

                sol1 = {
                "deg δ": x1,
                "deg γ": 0,
                "p": 0,
                "q": 0,
                "deg λ'": k2,
                "deg μ'": k4
                
                }
                print(f'       Solution is {sol1}')
                solutions.append(sol1)

            x3 = k1 - k2 - (x1 + x2)
            x4 = k3 - k4 - (x1 + x2)

            print(f'        p: {x3}')
            print(f'        q: {x4}\n\n')

            if (x3 < 0 or x4 < 0):
                print (f'REJECT: p or q is negative')
                continue

            
            if (x1 + 2 * x2 + x3 + x4 != k) or (x1 + x2 + x3 != k1 - k2) + (x1 + x2 + x4 != k3 - k4):
                print('Skipped')
                continue
            
            sol = {
                "deg δ": x1,
                "deg γ": x2,
                "p": x3,
                "q": x4,
                "deg λ'": k2,
                "deg μ'": k4
                }
            print(f' Solution is {sol}')
            solutions.append(sol)


        print('------------------------------ End of Gamma Loop ----------------------------------------\n')
    print('=============================== End of Delta Loop ========================================\n\n')  




    return solutions


In [10]:
# Examples: Correct Checked by Hand

# print(solveOne_new(2, 1, 0, 1, 1))
# print(solveOne_new(2, 1, 1, 1, 0))

# Ex1 by Professor

print(solveOne(3, 2, 0, 1, 0))


Start of iteration for Delta (from 0 to k):

======= The 0-th ITERATION FOR DELTA: ========

    Value of delta = 0
    Maximum Gamma = 3

    Start of iteration for Gamma (from 0 to ceil(floor(k - delta)/2) + 1): 

------- 0-th iteration of Gamma: --------

        Value of Gamma = 0
        p: 2
        q: 1


 Solution is {'deg δ': 0, 'deg γ': 0, 'p': 2, 'q': 1, "deg λ'": 0, "deg μ'": 0}
------- 1-th iteration of Gamma: --------

        Value of Gamma = 1
        p: 1
        q: 0


 Solution is {'deg δ': 0, 'deg γ': 1, 'p': 1, 'q': 0, "deg λ'": 0, "deg μ'": 0}
------- 2-th iteration of Gamma: --------

        Value of Gamma = 2
        p: 0
        q: -1


REJECT: p or q is negative
------------------------------ End of Gamma Loop ----------------------------------------

======= The 1-th ITERATION FOR DELTA: ========

    Value of delta = 1
    Maximum Gamma = 2

    Start of iteration for Gamma (from 0 to ceil(floor(k - delta)/2) + 1): 

------- 0-th iteration of Gamma: -------

In [11]:
def solveTwo(mu1, mu2, mu3, mu4):

    v1 = mu1 + mu2
    v2 = mu3 + mu4
    

    return [v1, v2]


In [12]:
def restrict_partitions(part1_list, part2_list, d_lam):
    """
    Filters two lists of partitions (part1_list, part2_list) so that
    only pairs (a, b) satisfy |a| + |b| = |lam|, where |.| is the degree (sum of parts).

    Returns:
        (filtered_part1, filtered_part2)
        where both are lists of partitions that can potentially combine to lam.
    """

    filtered_part1 = []
    filtered_part2 = []

    for a in part1_list:
        for b in part2_list:
            if sum(a) + sum(b) == d_lam:
                filtered_part1.append(a)
                filtered_part2.append(b)

    return filtered_part1, filtered_part2


In [13]:
# -----------------------------------------------------------------
#  three‑fold LR convolution – unchanged logic, corrected arguments
# -----------------------------------------------------------------
def lrcoef4(mu1, mu2, mu3, mu4, lam):

    print(f'--------- Calculation for term N^{lam}_{mu1}, {mu2}, {mu3}, {mu4}) ----------')

    d_mu1 = degree(mu1)
    d_mu2 = degree(mu2)
    d_mu3 = degree(mu3)
    d_mu4 = degree(mu4)
    d_lam = degree(lam)
    
    total1 = d_mu1 + d_mu2                # degree of the first intermediate partition
    total2 = d_mu3 + d_mu4                # degree of the second intermediate partition


    print(f'Relis on partitions v1, v2 such that:\n')
    print(f'deg of v1:{total1}')
    print(f'deg of v2:{total2}')

    parts_a = list(generate_partitions(total1))
    parts_b = list(generate_partitions(total2))
    parts_v1, parts_v2 = restrict_partitions(parts_a, parts_b, d_lam)

    print(f'\n Generate all such partitions: \n     V1 = {parts_a} \n       V2 = {parts_b} \n')

    # print(type(parts_v1[0]), parts_v2[0])

    print(f'========= SUMMATION START =========\n')

    s = 0
    i = 1
    for a in parts_a:
        print(f'\n---{i}-th iteration: ---\n')
        i += 1


        
        for b in parts_b:
            N1 = int(lrcoeff(a, mu1, mu2))   # c^{a}_{mu1,mu2}
            print(f' N1 = N^{a}_{mu1},{mu2} = {N1}')
            
            if N1 == 0: 
                print('Value zero')
                continue
    
            N2 = int(lrcoeff(b, mu3, mu4))   # c^{b}_{mu3,mu4}
            print(f'N2 = N^{b}_{mu3},{mu4} = {N2}')
            if N2 == 0:
                print('Value zero')
                continue
            N3 = int(lrcoeff(lam, a, b))           # c^{lam}_{a,b}
            print(f'N3 = N^{lam}_{a},{b} = {N3}')
            if N3 == 0:
                print('Value zero')
                continue
                
            s_term = N1 * N2 * N3

            print(f'Sum term: {s_term}')
            s += s_term
            print(f'Current Sum: {s}')
        
    
    print(f'\n======== Littlewood Richardson Coefficient (4 - order): \n N^{lam}_{mu1}, {mu2}, {mu3}, {mu4}) = {s} =========\n')
    return s


In [14]:
# # print(f'{lrcoef4((1,), (), (), (1,), (1,1))} \n done! \n')
# print(f'{lrcoef4((), (), (), (1,), (1,))} \n done! \n')
print(f'{lrcoef4((1,), (1,), (1,), (), (3,))}') 








--------- Calculation for term N^(3,)_(1,), (1,), (1,), ()) ----------
Relis on partitions v1, v2 such that:

deg of v1:2
deg of v2:1

 Generate all such partitions: 
     V1 = [(2,), (1, 1)] 
       V2 = [(1,)] 

========= SUMMATION START =========


---1-th iteration: ---

 N1 = N^(2,)_(1,),(1,) = 1
N2 = N^(1,)_(1,),() = 1
N3 = N^(3,)_(2,),(1,) = 1
Sum term: 1
Current Sum: 1

---2-th iteration: ---

 N1 = N^(1, 1)_(1,),(1,) = 1
N2 = N^(1,)_(1,),() = 1
N3 = N^(3,)_(1, 1),(1,) = 0
Value zero

======== Littlewood Richardson Coefficient (4 - order): 
 N^(3,)_(1,), (1,), (1,), ()) = 1 =========

1


In [15]:
def ones_partition(x):
    """Return the partition (1, 1, ..., 1) with x ones."""
    if x < 0:
        raise ValueError("x must be nonnegative")
    return tuple(1 for _ in range(x))


In [16]:
def CalcSoc(k, lam, lamP, mu, muP):
    """
    k1 = |lam|,   k3 = |mu|
    """

    k1 = degree(lam)
    k2 = degree(lamP)
    k3 = degree(mu)
    k4 = degree(muP)

    print (f'Values of the degrees of: \n lambda, delta lambda Prime, mu prime, deg mu Prime, {k1, k2, k3, k4} \n')
    
    L = solveOne(k, k1, k2, k3, k4)

    print(f'Solution degrees for Delta, Gamma, 1^p and 1^q are:\n')
    print(f'    {L}\n\n')

    print('++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++\n')
    print('++++++++++++++++++ START of Total Summation ++++++++++++++++++\n')
    print('++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++\n')

    
    tot_sum = 0

    for sol in L:
        print(f'For {sol}: \n Lists of possible partitions for:\n')

        d_list = list(generate_partitions(sol["deg δ"]))
        g_list = list(generate_partitions(sol["deg γ"]))
        print(f'    Delta: \n {d_list}')
        print(f'    Gamma: \n {g_list}')

        print("As well as:")
        
        p = ones_partition(sol["p"])
        q = ones_partition(sol["q"])
        print(f'    1^p = {p}')
        print(f'    1^q = {q} \n\n')

        print("\n=========== START OF LOOP IN DELTA FOR EACH PRODUCT ========\n")
        for d in d_list:
            print(f'-------------- START OF LOOP IN GAMMA --------------')
            for g in g_list:
                print("\n=========== START OF LOOP FOR EACH PRODUCT ========\n")
                print(f'solution partitions: d:{d}, g: {g}, 1^p:{p}, 1^q{q}\n')


                # first factor uses the *full* partition lam
                term1 = lrcoef4(lamP, g, d, p, lam)
                print(f'term (N^lam) = {term1}')

                # second factor uses the *full* partition mu
                term2 = lrcoef4(muP, g, d, q, mu)
                print(f'term (N^mu) = {term2}')

                tot_sum += term1 * term2
            print('------------------------- END ---------------------------\n')
        
        print('============================ END ==================================')
    print(f'\n <<<<<<<<<<<<< {k + 1}-th Socle Filtration is : {tot_sum} >>>>>>>>>>>>>\n\n')
    return tot_sum


In [17]:
def Master1(k, lam, lamP, mu, muP, ex_string = ''):
    print(f'\n--------{ex_string}-------- \n')
    print(f'''|----------- Calculation for {k+1}-th soc given partitions: ----------| \n 
            lambda = {lam} \n
            lambda Prime = {lamP} \n
            mu = {mu} \n 
            mu Prime = {muP} \n
            k-value = {k} \n''')



    #print(f'------ Calculation of suitable partitions delta, gamma, 1^p, 1^q ------ \n')
    #print(solveOne(k, degree(lam), degree(lamP), degree(mu), degree(muP)))
    print(CalcSoc(k, lam, lamP, mu, muP))


# My Example with Master

# Master1(3, (1, 2), (1, 1), (2,), (), 'My Example')
# My Example



# Ex 1

# Master1(1, (6,), (4,2), (1,1,1,1,1), (5,), 'Example 1')


# Ex 2

# Master1(1, (1,), (), (1,), (), 'Example 2')


# Ex 3

# Master1(2, (1,), (), (1,), ())

# Ex 4

# Master1(2, (1,1), (), (1,), (), 'Example 4') # also works for k =3

#Ex 5

# Master1(2, (1,1), (1,), (1,), (), 'Example 5') # CHECKS FINE!!!

# Master1(3, (1,1), (), (1,), (), 'Example 6')


# new example

# Master1(2, (1,), (), (1,), (1,), "new example")

#Master1(4, (1,1), (), (1,1), ())

# Big exampple:

k = 2
lam = (4,)
mu = ()

lamP = (2,)
muP = ()
#
Master1(k, lam, lamP, mu, muP)




---------------- 

|----------- Calculation for 3-th soc given partitions: ----------| 
 
            lambda = (4,) 

            lambda Prime = (2,) 

            mu = () 
 
            mu Prime = () 

            k-value = 2 

Values of the degrees of: 
 lambda, delta lambda Prime, mu prime, deg mu Prime, (4, 2, 0, 0) 

Start of iteration for Delta (from 0 to k):

======= The 0-th ITERATION FOR DELTA: ========

    Value of delta = 0
    Maximum Gamma = 2

    Start of iteration for Gamma (from 0 to ceil(floor(k - delta)/2) + 1): 

------- 0-th iteration of Gamma: --------

        Value of Gamma = 0
        p: 2
        q: 0


 Solution is {'deg δ': 0, 'deg γ': 0, 'p': 2, 'q': 0, "deg λ'": 2, "deg μ'": 0}
------- 1-th iteration of Gamma: --------

        Value of Gamma = 1
        p: 1
        q: -1


REJECT: p or q is negative
------------------------------ End of Gamma Loop ----------------------------------------

======= The 1-th ITERATION FOR DELTA: ========

    Value of del